# Zero-Shot Emotion Recognition Baseline: InstructBLIP 7B on LAI-GAI

This self-contained Jupyter notebook performs zero-shot emotion classification on the **LAI-GAI dataset** (847 culturally diverse images across 12 affective classes) using **InstructBLIP (Salesforce/instructblip-vicuna-7b)**.

### Workflow Steps:
1. **Environment Setup**: Install required dependencies (`transformers`, `accelerate`, `bitsandbytes`, `datasets`).
2. **Dataset Acquisition**: Download `ACG-AMU/LAI-GAI` from Hugging Face and save labeled images (`<id>_<emotion>.png`).
3. **Data Loading & Regex Parsing**: Extract ground-truth emotion labels from filenames.
4. **Model Loading**: Initialize `Salesforce/instructblip-vicuna-7b` in 4-bit NF4 with explicit device mapping.
5. **Zero-Shot Inference**: Prompt the model image-by-image and map raw responses to candidate emotions.
6. **Evaluation & Metrics**: Stream results to CSV, compute overall accuracy, per-class report, and confusion matrix.


In [ ]:
# Step 1: Install required dependencies
!pip install -q --upgrade transformers accelerate bitsandbytes datasets Pillow pandas scikit-learn seaborn matplotlib tqdm


In [ ]:
# Step 2: System Imports and Environment Configuration
import os
import re
import time
import csv
import torch
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from datasets import load_dataset
from transformers import (
    InstructBlipProcessor,
    InstructBlipForConditionalGeneration,
    BitsAndBytesConfig,
)

# Working directories for Kaggle or local environments
BASE_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
DATA_DIR = os.path.join(BASE_DIR, 'data', 'lai_gai_labeled_images')
RESULTS_DIR = os.path.join(BASE_DIR, 'results')
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Allocated VRAM: {torch.cuda.memory_allocated() / (1024**3):.2f} GB')
print(f'Data directory: {DATA_DIR}')
print(f'Results directory: {RESULTS_DIR}')


In [ ]:
# Step 3: Download LAI-GAI Dataset from Hugging Face and Format Filenames
def download_and_format_laigai(output_dir: str):
    existing_files = [f for f in os.listdir(output_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    if len(existing_files) >= 847:
        print(f'[✓] LAI-GAI dataset already downloaded at "{output_dir}" ({len(existing_files)} images found). Skipping download.')
        return

    print('Downloading ACG-AMU/LAI-GAI dataset from Hugging Face Hub...')
    ds = load_dataset('ACG-AMU/LAI-GAI')
    train_split = ds['train']
    print(f'Found {len(train_split)} images in dataset split. Saving to disk...')

    saved_count = 0
    for idx, item in enumerate(tqdm(train_split, desc='Saving LAI-GAI Images')):
        emotion_label = item.get('emotion', item.get('target_emotion', 'unknown'))
        emotion_label = str(emotion_label).strip().capitalize()
        img = item['image'].convert('RGB')
        
        img_filename = f'{idx:03d}_{emotion_label}.png'
        img_path = os.path.join(output_dir, img_filename)
        img.save(img_path)
        saved_count += 1

    print(f'[✓] Successfully downloaded and saved {saved_count} images to "{output_dir}".')

download_and_format_laigai(DATA_DIR)


In [ ]:
# Step 4: Load LAI-GAI Dataset via Regex
def load_lai_gai_dataset(dataset_dir: str) -> tuple[list[dict], list[str]]:
    """
    Extracts emotion labels from filenames using regex matching:
    Format: <index>_<emotion>.png (e.g. 000_Craving.png, 013_Nurturant love.png)
    """
    pattern = re.compile(r"^\d+_(.+)\.(?:png|jpg|jpeg)$", re.IGNORECASE)
    dataset_items = []
    all_labels = set()

    for fname in sorted(os.listdir(dataset_dir)):
        match = pattern.match(fname)
        if match:
            raw_emotion = match.group(1).strip()
            emotion = raw_emotion.capitalize()
            full_path = os.path.join(dataset_dir, fname)
            dataset_items.append({
                'image_path': full_path,
                'filename': fname,
                'ground_truth': emotion,
            })
            all_labels.add(emotion)

    candidate_emotions = sorted(list(all_labels))
    return dataset_items, candidate_emotions

dataset, candidate_emotions = load_lai_gai_dataset(DATA_DIR)
print(f'Total loaded images: {len(dataset)}')
print(f'Number of emotion classes: {len(candidate_emotions)}')
print(f'Candidate emotions: {candidate_emotions}')

df_dist = pd.DataFrame(dataset)
print('\nClass counts in dataset:')
print(df_dist['ground_truth'].value_counts())


In [ ]:
# Step 5: Load InstructBLIP 7B with 4-bit NF4 Quantization
MODEL_ID = 'Salesforce/instructblip-vicuna-7b'

print(f'Initializing processor and 4-bit model: {MODEL_ID}...')
processor = InstructBlipProcessor.from_pretrained(MODEL_ID)

device_map = {"": 0} if torch.cuda.is_available() else "auto"

model = InstructBlipForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map=device_map,
)

model.eval()

print('[✓] Model and processor successfully loaded onto GPU!')
if torch.cuda.is_available():
    print(f'VRAM currently allocated: {torch.cuda.memory_allocated() / (1024**3):.2f} GB')


In [ ]:
# Step 6: Zero-Shot Prompt & Structured Output Parser
def build_zero_shot_prompt(candidate_emotions: list[str]) -> str:
    """
    Constructs a direct VQA prompt matching InstructBLIP's QA format.
    """
    labels_str = ', '.join(candidate_emotions)
    return (
        f'Question: What emotion is conveyed by this image? '
        f'Choose one single emotion from the following list: {labels_str}. '
        f'Provide only the emotion word.\n'
        f'Answer:'
    )

def parse_model_prediction(raw_output: str, candidate_emotions: list[str]) -> str:
    """
    Extracts and maps raw MLLM text to one of the exact candidate emotions.
    Handles exact matches, compound emotion labels ('Attachment love', 'Nurturant love'),
    and substring matching.
    """
    if not raw_output or not raw_output.strip():
        return candidate_emotions[0]

    cleaned = raw_output.replace('\\_', '_').strip()
    cleaned_lower = cleaned.lower()

    # 1. Exact whole-phrase word boundary match
    for emotion in candidate_emotions:
        if re.search(rf'\b{re.escape(emotion)}\b', cleaned, re.IGNORECASE):
            return emotion

    # 2. Compound label resolution (handles 'Love' / 'Attachment' / 'Nurturant')
    if 'love' in cleaned_lower:
        if 'nurtur' in cleaned_lower:
            return 'Nurturant love'
        else:
            return 'Attachment love'

    # 3. Substring match for keywords (> 3 chars)
    for emotion in candidate_emotions:
        words = [w.lower() for w in emotion.split() if len(w) > 3]
        if any(w in cleaned_lower for w in words):
            return emotion

    # 4. JSON fallback if model produced JSON
    json_match = re.search(r'\{[^{}]*"emotion"\s*:\s*"([^"]+)"[^{}]*\}', cleaned, re.IGNORECASE)
    if json_match:
        extracted = json_match.group(1).strip()
        matched = next((e for e in candidate_emotions if e.lower() == extracted.lower()), None)
        if matched:
            return matched

    # 5. Default fallback
    return candidate_emotions[0]

def run_zero_shot_inference(image: Image.Image, prompt_text: str) -> str:
    """
    Executes inference with InstructBLIP and slices only the newly generated tokens.
    """
    inputs = processor(images=image, text=prompt_text, return_tensors='pt').to(device)
    input_len = inputs['input_ids'].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=30,
            repetition_penalty=1.05,
            do_sample=False,
        )

    # Slice past prompt tokens to get clean generated response
    if outputs.shape[1] > input_len:
        new_tokens = outputs[:, input_len:]
        raw_text = processor.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()
    else:
        raw_text = processor.batch_decode(outputs, skip_special_tokens=True)[0].strip()
        if raw_text.startswith(prompt_text):
            raw_text = raw_text[len(prompt_text):].strip()

    del inputs, outputs
    torch.cuda.empty_cache()
    return raw_text


In [ ]:
# Step 7: Run End-to-End Zero-Shot Evaluation Loop Across All Images
csv_filename = os.path.join(RESULTS_DIR, 'laigai_zero_shot_instructblip_7b.csv')
zero_shot_prompt = build_zero_shot_prompt(candidate_emotions)

print(f'Zero-Shot Evaluation Prompt:\n{zero_shot_prompt}\n')
print(f'Evaluating {len(dataset)} images... Results saved to: {csv_filename}')

correct_predictions = 0
total_evaluated = 0
results_log = []
start_time = time.time()

with open(csv_filename, mode='w', newline='', encoding='utf-8') as csv_file:
    csv_writer = csv.writer(csv_file)
    csv_writer.writerow([
        'image_path',
        'ground_truth',
        'predicted_emotion',
        'is_correct',
        'raw_output',
        'elapsed_sec',
    ])

    pbar = tqdm(dataset, desc='Evaluating Images')
    for item in pbar:
        img_path = item['image_path']
        gt = item['ground_truth']
        step_t0 = time.time()

        try:
            raw_img = Image.open(img_path).convert('RGB')
            raw_img.thumbnail((512, 512), Image.LANCZOS)

            raw_response = run_zero_shot_inference(raw_img, zero_shot_prompt)
            prediction = parse_model_prediction(raw_response, candidate_emotions)
        except Exception as err:
            print(f'[Error processing {img_path}]: {err}')
            raw_response = str(err)
            prediction = candidate_emotions[0]

        step_duration = round(time.time() - step_t0, 3)
        is_match = int(prediction.strip().lower() == gt.strip().lower())
        correct_predictions += is_match
        total_evaluated += 1

        row = [img_path, gt, prediction, is_match, raw_response, step_duration]
        csv_writer.writerow(row)
        csv_file.flush()
        results_log.append(row)

        current_acc = (correct_predictions / total_evaluated) * 100
        pbar.set_postfix({
            'Acc': f'{current_acc:.2f}%',
            'Correct': f'{correct_predictions}/{total_evaluated}',
        })

total_time = time.time() - start_time
overall_acc = (correct_predictions / total_evaluated) * 100 if total_evaluated > 0 else 0
print('\n' + '=' * 60)
print('Zero-Shot Evaluation Complete!')
print(f'Total Images Evaluated: {total_evaluated}')
print(f'Correct Predictions  : {correct_predictions}')
print(f'Final Accuracy       : {overall_acc:.2f}%')
print(f'Total Time           : {total_time:.2f} seconds ({total_time/total_evaluated:.2f}s per image)')
print('=' * 60)


In [ ]:
# Step 8: Detailed Classification Report and Confusion Matrix Plot
df_results = pd.read_csv(csv_filename)
y_true = df_results['ground_truth']
y_pred = df_results['predicted_emotion']

print('\n--- Detailed Per-Class Classification Report ---')
report = classification_report(y_true, y_pred, labels=candidate_emotions, zero_division=0)
print(report)

# Plot normalized confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=candidate_emotions, normalize='true')
plt.figure(figsize=(12, 10))
sns.heatmap(
    cm,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    xticklabels=candidate_emotions,
    yticklabels=candidate_emotions,
)
plt.title('InstructBLIP-7B Zero-Shot Normalized Confusion Matrix (LAI-GAI)', fontsize=14)
plt.xlabel('Predicted Emotion', fontsize=12)
plt.ylabel('Ground Truth Emotion', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
cm_plot_path = os.path.join(RESULTS_DIR, 'confusion_matrix_instructblip_zero_shot.png')
plt.savefig(cm_plot_path, dpi=300)
plt.show()
print(f'Saved confusion matrix plot to: {cm_plot_path}')
